# Figure 2C — DE Phosphosite Cluster Trajectory Plots

In [11]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
import os
from scipy import stats

In [12]:
# directory and data file settings
psites_f = "/ix/djishnu/Aaron_F/proteomics_proj/Data/phosphoproteomics/TMT_control_centered_log2fc.csv"
desites_f = "/ix/djishnu/Aaron_F/proteomics_proj/Data/phosphoproteomics/significant_phospho_pairwise_ids_and_flanks.csv"
cluster_f = "/ix/djishnu/Aaron_F/proteomics_proj/Results/pairwise_DE_sites_analysis/out_files/pairwise_DEsites_9_clusters_ordered.csv"

out_dir = "/ix/djishnu/Aaron_F/proteomics_proj/Results/Figures/Fig2c"

os.makedirs(out_dir, exist_ok=True)
os.makedirs(f"{out_dir}/figures", exist_ok=True)
os.makedirs(f"{out_dir}/out_files", exist_ok=True)
os.chdir(out_dir)

In [13]:
# load DE sites:
desites = pd.read_csv(desites_f)['id'].tolist()

# load prot. data and subset to desites
prot_df = pd.read_csv(psites_f)
psite_id_map = prot_df[["id", "geneSymbol"]].copy()
prot_df.set_index("id", inplace=True)
prot_df.drop(columns=["geneSymbol"], inplace=True)
prot_df = prot_df.loc[
    desites,
].copy()
prot_df.reset_index(inplace=True)

cluster_info = pd.read_csv(cluster_f)
print(cluster_info.head())

                              id  cluster geneSymbol cluster_color
0  P83741-3_S2282s_1_0_2280_2282        1       Wnk1       #f77189
1    E9PWW9_S1312s_1_1_1312_1312        1       Rsf1       #f77189
2       O09047_T456t_1_1_456_456        1      C3ar1       #f77189
3  O09047_T459tT462t_2_1_452_462        1      C3ar1       #f77189
4       Q3UZA1_S380s_1_0_378_380        1      Rcsd1       #f77189


In [14]:
prot_df = prot_df.merge(
    cluster_info[["id", "cluster", "geneSymbol", "cluster_color"]],
    left_on="id",
    right_on="id",
    how="inner"
)

prot_df = prot_df.set_index("id")

## Per-cluster IL6 vs IL10 trajectory plots (mean ± SEM)

In [15]:
time_order = ["0m", "20m", "45m", "90m"]

unstim_cols = [c for c in prot_df.columns if "PBS_0" in c]
il6_cols    = [c for c in prot_df.columns if "IL6" in c]
il10_cols   = [c for c in prot_df.columns if "IL10" in c]

clusters = sorted(prot_df["cluster"].unique())

# ---- Define consistent cytokine colors ----
IL6_COLOR = "#d62728"    # red
IL10_COLOR = "#1f77b4"   # blue

for clus in clusters:

    cluster_df = prot_df[prot_df["cluster"] == clus]

    def compute_summary(df, cytokine_cols):

        means = []
        sems = []

        for t in time_order:

            if t == "0m":
                time_cols = unstim_cols
            else:
                time_cols = [c for c in cytokine_cols if t.rstrip("m") in c]

            per_gene = df[time_cols].mean(axis=1)

            means.append(per_gene.mean())
            sems.append(stats.sem(per_gene))

        return np.array(means), np.array(sems)

    il6_mean, il6_sem = compute_summary(cluster_df, il6_cols)
    il10_mean, il10_sem = compute_summary(cluster_df, il10_cols)

    # ---- Plot ----
    plt.figure(figsize=(5,4))
    x = np.arange(len(time_order))

    # IL6
    plt.plot(x, il6_mean, marker="o", color=IL6_COLOR, label="IL6")
    plt.fill_between(
        x,
        il6_mean - il6_sem,
        il6_mean + il6_sem,
        color=IL6_COLOR,
        alpha=0.25
    )

    # IL10
    plt.plot(x, il10_mean, marker="o", color=IL10_COLOR, label="IL10")
    plt.fill_between(
        x,
        il10_mean - il10_sem,
        il10_mean + il10_sem,
        color=IL10_COLOR,
        alpha=0.25
    )

    plt.xticks(x, time_order)
    plt.xlabel("Time")
    plt.ylabel("Expression")
    plt.title(f"Cluster {clus} (n={len(cluster_df)})")
    plt.legend()
    plt.tight_layout()

    plt.savefig(f"{out_dir}/figures/cluster_{clus}_trajectory.png", dpi=300)
    plt.savefig(f"{out_dir}/figures/cluster_{clus}_trajectory.svg", dpi=300)
    plt.close()

print("Cluster trajectory plots saved.")

Cluster trajectory plots saved.
